<a href="https://colab.research.google.com/github/njones61/xslope/blob/main/notebooks/xslope_design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XSLOPE - Slope Design

This notebook finds the critical slope angle that produces a target factor of safety using the limit equilibrium method. It sweeps a range of slope angles, runs an automated circular search for each, and interpolates to find the angle corresponding to the design FS.

## Install xslope and import functions

In [ ]:
%%capture
!pip install xslope

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

from xslope.fileio import load_slope_data, build_ground_surface
from xslope.plot import plot_solution, plot_inputs
from xslope.search import circular_search

## Upload Excel Template

In [ ]:
from google.colab import files
upload = files.upload()
file_name = list(upload.keys())[0]

# See if uploaded file is a zip archive. If so, unzip it
if file_name.endswith('.zip'):
  import zipfile
  with zipfile.ZipFile(file_name, 'r') as zip_ref:
    zip_ref.extractall()
    extracted_files = zip_ref.namelist()

    excel_file_found = False
    for f in extracted_files:
      if f.endswith('.xlsx'):
        file_name = f
        excel_file_found = True
        print(f"Found Excel file: {file_name}")
        break

    if not excel_file_found:
        print("Error: No .xlsx file found in the uploaded archive.")
        file_name = None

## Load slope data

In [ ]:
slope_data = load_slope_data(file_name)
plot_inputs(slope_data, mode='lem', save_png=False)

## Design Parameters

In [ ]:
# @title Select Options {"run":"auto"}
method = "spencer" # @param ["oms","bishop","janbu","corps_engineers","lowe_karafiath","spencer"]

beta1 = 25  # @param {"type":"number"}
beta2 = 35  # @param {"type":"number"}
design_fs = 1.5  # @param {"type":"number"}
toe_index = 1  # @param {"type":"integer"}
slope_index = 2  # @param {"type":"integer"}

save_png = True # @param {"type":"boolean"}

## Sweep slope angles

In [ ]:
from numpy import save


betas = np.linspace(beta1, beta2, num=10)
fs_results = np.zeros_like(betas)
for i, beta in enumerate(betas):

  profile = slope_data['profile_lines'][0]['coords']
  x_toe, y_toe = profile[toe_index]
  x_top, y_top = profile[slope_index]

  # Calculate new x_top from beta: tan(beta) = (y_top - y_toe) / (x_top - x_toe)
  x_top_new = x_toe + (y_top - y_toe) / math.tan(math.radians(beta))
  slope_data['profile_lines'][0]['coords'][slope_index] = (x_top_new, y_top)
  slope_data['ground_surface'] = build_ground_surface(slope_data['profile_lines'])

  print(f"Slope angle: {beta:.1f}\u00b0, toe: ({x_toe}, {y_toe}), slope point: ({x_top_new:.2f}, {y_top})")

  fs_cache, converged, search_path, circle_cache = circular_search(slope_data, method)
  fs_results[i] = fs_cache[0]['FS']

## Interpolate critical slope angle

In [ ]:
# Check that design_fs is within the computed range
fs_min, fs_max = fs_results.min(), fs_results.max()
if not (fs_min <= design_fs <= fs_max):
    print(f"Design FS={design_fs} is outside the computed range [{fs_min:.3f}, {fs_max:.3f}].")
    print(f"Adjust beta1/beta2 so the range brackets FS={design_fs} and re-run.")
else:
    critical_slope_angle = float(np.interp(design_fs, fs_results[::-1], betas[::-1]))
    print(f"Critical slope angle for FS={design_fs}: {critical_slope_angle:.2f}\u00b0")

## Verify at critical slope angle

In [ ]:
# Redo the analysis at the critical slope angle to confirm FS is close to design_fs
profile = slope_data['profile_lines'][0]['coords']
x_toe, y_toe = profile[toe_index]
x_top, y_top = profile[slope_index]
x_top_new = x_toe + (y_top - y_toe) / math.tan(math.radians(critical_slope_angle))
slope_data['profile_lines'][0]['coords'][slope_index] = (x_top_new, y_top)
slope_data['ground_surface'] = build_ground_surface(slope_data['profile_lines'])

fs_cache, converged, search_path, circle_cache = circular_search(slope_data, method)
fs_at_critical = fs_cache[0]['FS']
print(f"FS at critical slope angle ({critical_slope_angle:.2f}\u00b0): {fs_at_critical:.3f}")

critical_surface = fs_cache[0]
slice_df = critical_surface['slices']
failure_surface = critical_surface['failure_surface']
results = critical_surface['solver_result']
plot_solution(slope_data, slice_df, failure_surface, results, save_png=save_png, file_name="design_results.png")

## Plot FS vs Slope Angle

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(betas, fs_results, marker='o')

# Horizontal line at FS = design_fs
ax.axhline(y=design_fs, color='r', linestyle='--', linewidth=0.8, label=f'FS = {design_fs}')

# Vertical line at critical slope angle
ax.axvline(x=critical_slope_angle, color='gray', linestyle='--', linewidth=0.8,
           label=f'\u03b2 = {critical_slope_angle:.1f}\u00b0')

# Mark the intersection point
ax.plot(critical_slope_angle, fs_at_critical, 's', color='r', markersize=8, zorder=5)

ax.set_xlabel('Slope Angle (degrees)')
ax.set_ylabel('Factor of Safety')
ax.set_title('Factor of Safety vs Slope Angle')
ax.legend()
ax.grid()
plt.tight_layout()
if save_png:
    plt.savefig('fs_vs_slope_angle.png')
plt.show()